# Customer Analytics: Segmentation, Retention & Revenue Concentration

**Notebook 01 of 03** · UCI Online Retail Dataset · Dec 2010 – Dec 2011

---

**Business context:** A UK-based online gift retailer wants to understand its customer base — who the most valuable customers are, where they lose them, and how revenue is distributed. This notebook builds the analytical foundation; the findings here feed directly into a SQL deep-dive (Notebook 02) and an A/B test simulation (Notebook 03).

**What this notebook covers:**

| # | Section | Question it answers |
|---|---------|-------------------|
| 1 | Data Cleaning | What does the raw data look like, and what needs to go before we can trust it? |
| 2 | RFM Segmentation | Can we group customers into meaningful behavioural segments? |
| 3 | Cohort Retention | When do we lose customers — early or late in their lifecycle? |
| 4 | Revenue Concentration | Does a small group drive most of the revenue, or is it broadly distributed? |

> **How to read this notebook:** Each section begins with a question, walks through the analysis, and ends with a *Findings* block summarising what the data shows and what it implies for the next section. Cluster labels (C0–C3) assigned in Section 2 are used consistently across all three notebooks.

---

## 1. Data Cleaning

The dataset has 541,909 transaction rows across 8 fields. Before any analysis, we need to understand the data quality and remove records that would distort behavioural metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ── Visual style ─────────────────────────────────────────────
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3']
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#333333',
    'axes.labelcolor': '#333333',
    'text.color': '#333333',
    'xtick.color': '#555555',
    'ytick.color': '#555555',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.grid': False,
    'figure.dpi': 120,
})

In [ ]:
df = pd.read_csv("online_retail.csv")
print(f"Raw dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

### Initial data inspection

Before deciding what to remove, let's check for missing values and obvious anomalies.

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'Missing': missing, '% of total': missing_pct}).query("Missing > 0")

~25% of rows have no `CustomerID` — these cannot be attributed to any customer and are unusable for behavioural analysis. `Description` has a small number of nulls, but we don't use this field for segmentation.

Let's also check for cancellations and invalid quantities:

In [ ]:
cancelled = df['InvoiceNo'].astype(str).str.startswith('C').sum()
neg_qty = (df['Quantity'] <= 0).sum()
neg_price = (df['UnitPrice'] <= 0).sum()

print(f"Cancelled orders (InvoiceNo starts with 'C'): {cancelled:,}")
print(f"Non-positive Quantity: {neg_qty:,}")
print(f"Non-positive UnitPrice: {neg_price:,}")

### Apply filters

Three categories of records are removed:

| Filter | Reason | Rows affected |
|--------|--------|--------------|
| Missing `CustomerID` | Cannot attribute to a customer | ~135k |
| `InvoiceNo` starting with `C` | Cancellation records, not real purchases | ~9.3k |
| `Quantity ≤ 0` or `UnitPrice ≤ 0` | Data entry errors or adjustments | overlaps with above |

A `TotalPrice` column (`Quantity × UnitPrice`) is added as the revenue metric used throughout.

In [ ]:
# ── Cleaning ─────────────────────────────────────────────────
df = df.dropna(subset=['CustomerID'])
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

print(f"Clean dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Unique customers: {df['CustomerID'].nunique():,}")
print(f"Date range: {df['InvoiceDate'].min().date()} → {df['InvoiceDate'].max().date()}")

---

## 2. RFM Segmentation

**Question:** Can we group 4,000+ customers into a small number of meaningful behavioural segments that the business can act on differently?

RFM reduces each customer's full transaction history to three numbers:

| Feature | Definition | Better = |
|---------|-----------|----------|
| **Recency** | Days since last purchase (relative to dataset end + 1 day) | Lower |
| **Frequency** | Number of distinct orders | Higher |
| **Monetary** | Total lifetime spend (£) | Higher |

In [ ]:
# ── Build RFM table ───────────────────────────────────────────
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                                     # Frequency
    'TotalPrice': 'sum'                                         # Monetary
})
rfm.columns = ['Recency', 'Frequency', 'Monetary']

print(f"RFM table: {rfm.shape[0]:,} customers")
rfm.describe().round(1)

### Standardisation

Features are standardised to zero mean and unit variance before clustering. This is necessary because Monetary (range: £3 – £280k) has a much larger raw scale than Recency or Frequency — without standardisation, KMeans' Euclidean distance would be dominated by Monetary alone.

> **Note on skewness:** Monetary and Frequency are right-skewed (a few extreme customers pull the mean far above the median). A log-transform before standardisation is a common alternative that compresses the tail. We proceed with standard scaling here because the resulting clusters produce interpretable, business-meaningful segments — but this is a modelling choice worth revisiting if the clusters were less distinct.

In [ ]:
# ── Standardise ───────────────────────────────────────────────
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

### Dimensionality reduction (PCA) — for visualisation only

PCA compresses the 3D RFM space into 2D so we can plot customers on a scatter chart. **Important:** clustering is performed on the full 3-feature standardised space, not on the PCA output. Reducing to 2D before clustering would discard variance and distort cluster boundaries.

In [ ]:
pca = PCA(n_components=2)
rfm_pca = pca.fit_transform(rfm_scaled)

var_explained = pca.explained_variance_ratio_
print(f"PC1 explains {var_explained[0]:.1%} of variance")
print(f"PC2 explains {var_explained[1]:.1%} of variance")
print(f"Total:        {var_explained.sum():.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(rfm_pca[:, 0], rfm_pca[:, 1], alpha=0.25, s=12, color=PALETTE[0], edgecolors='none')
ax.set_xlabel(f"PC1 ({var_explained[0]:.0%} variance)")
ax.set_ylabel(f"PC2 ({var_explained[1]:.0%} variance)")
ax.set_title("Customer Distribution in PCA Space (before clustering)")
plt.tight_layout()
plt.show()

The bulk of customers form a dense cluster near the origin, with a long tail stretching to the upper-right — these are likely high-frequency, high-spend customers. This suggests the data has natural groupings that KMeans should be able to separate.

### Choosing K

We use two methods to select the number of clusters:
1. **Elbow method** — plots inertia (within-cluster sum of squares) vs. K; we look for the "elbow" where adding clusters yields diminishing returns.
2. **Silhouette score** — measures how similar each point is to its own cluster vs. the nearest other cluster. Higher = better-defined clusters.

In [ ]:
# ── Elbow + Silhouette ────────────────────────────────────────
K_range = range(2, 11)
inertia = []
silhouette_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled)
    inertia.append(km.inertia_)
    silhouette_scores.append(silhouette_score(rfm_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Elbow
ax1.plot(K_range, inertia, marker='o', color=PALETTE[0], linewidth=2, markersize=6)
ax1.axvline(x=4, linestyle='--', color=PALETTE[3], alpha=0.6, label='K = 4')
ax1.set_xlabel("Number of Clusters (K)")
ax1.set_ylabel("Inertia")
ax1.set_title("Elbow Method")
ax1.legend()

# Silhouette
ax2.bar(K_range, silhouette_scores, color=[PALETTE[3] if k == 4 else PALETTE[0] for k in K_range], alpha=0.8)
ax2.set_xlabel("Number of Clusters (K)")
ax2.set_ylabel("Silhouette Score")
ax2.set_title("Silhouette Score by K")

for k, s in zip(K_range, silhouette_scores):
    ax2.text(k, s + 0.005, f"{s:.3f}", ha='center', fontsize=8, color='#555555')

plt.tight_layout()
plt.show()

**K = 4 is selected.** The elbow curve flattens after K=4, and the silhouette score at K=4 is near the peak — adding a 5th cluster doesn't meaningfully improve either metric. More importantly, K=4 produces segments with distinct, interpretable business meaning (verified below). A purely mechanical elbow is a starting point; the chosen K must also make analytical sense.

### Fit KMeans (K=4)

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

In [ ]:
# ── Cluster scatter (PCA space) ────────────────────────────────
cluster_colors = {0: PALETTE[0], 1: PALETTE[1], 2: PALETTE[3], 3: PALETTE[2]}
cluster_labels = {0: 'C0 — Regular', 1: 'C1 — At-Risk', 2: 'C2 — Whale/B2B', 3: 'C3 — High-Value Loyal'}

fig, ax = plt.subplots(figsize=(9, 6))
for c in sorted(rfm['Cluster'].unique()):
    mask = rfm['Cluster'] == c
    ax.scatter(rfm_pca[mask, 0], rfm_pca[mask, 1],
               alpha=0.4, s=15, color=cluster_colors[c],
               label=cluster_labels[c], edgecolors='none')

ax.set_xlabel(f"PC1 ({var_explained[0]:.0%} variance)")
ax.set_ylabel(f"PC2 ({var_explained[1]:.0%} variance)")
ax.set_title("Customer Segments in PCA Space (K=4)")
ax.legend(loc='upper right', framealpha=0.9, fontsize=9)
plt.tight_layout()
plt.show()

### Cluster profiles

In [ ]:
# ── Profile table ──────────────────────────────────────────────
profile = rfm.groupby('Cluster').agg(
    Customers=('Recency', 'size'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean'),
    Median_Monetary=('Monetary', 'median')
).round(1)

profile.index = [cluster_labels.get(c, c) for c in profile.index]
profile

In [ ]:
# ── Radar-style bar chart for cluster comparison ──────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
metrics = ['Avg_Recency', 'Avg_Frequency', 'Avg_Monetary']
titles = ['Avg Recency (days) ↓ lower = better', 'Avg Frequency (orders) ↑', 'Avg Monetary (£) ↑']
profile_raw = rfm.groupby('Cluster').agg(
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean'),
).round(1)

for ax, metric, title in zip(axes, metrics, titles):
    values = profile_raw[metric]
    bars = ax.barh(
        [cluster_labels[c] for c in values.index],
        values,
        color=[cluster_colors[c] for c in values.index],
        alpha=0.85
    )
    ax.set_title(title, fontsize=10)
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + max(values)*0.02, bar.get_y() + bar.get_height()/2,
                f'{val:,.0f}', va='center', fontsize=9, color='#555555')

plt.tight_layout()
plt.show()

### Findings

**Four distinct behavioural archetypes emerge:**

| Cluster | Label | Size | Behavioural signature |
|---------|-------|------|-----------------------|
| C0 | **Regular** | ~3,054 | Moderate recency (~44d), moderate frequency (~3.7 orders), moderate spend. The largest group — active but not deeply engaged. |
| C1 | **At-Risk / Lapsed** | ~1,067 | High recency (~248d), low frequency (~1.6 orders). These customers haven't purchased in ~8 months — well beyond any normal purchase cycle. |
| C2 | **Whale / B2B** | ~13 | Extremely low recency (~7d), very high frequency (~82 orders), very high spend. Almost certainly wholesale or B2B accounts — not typical retail buyers. |
| C3 | **High-Value Loyal** | ~204 | Low recency (~16d), high frequency (~22 orders), high spend. The core retail revenue drivers. |

**Key observations:**

C0 and C1 have similar low-to-moderate frequency, but completely different recency. C0's ~44-day recency is consistent with an active purchase cycle; C1's ~248-day recency is structural absence, not a delayed purchase. The precise inter-purchase gap analysis that quantifies this distinction is in Notebook 02.

C2's extreme values (82 orders, avg £8.5k spend) are inconsistent with retail purchasing — these are almost certainly B2B accounts and need to be handled separately in downstream analysis.

> **What this tells us so far:** Customers are not uniform. But RFM is a static snapshot — it tells us *where* each customer is, not *when* or *how fast* they got there. To understand when we lose customers, we need a time-based view → Section 3.

### Export cluster labels

The `CustomerID → Cluster` mapping is exported for use by Notebook 02 (loaded into DuckDB for SQL analysis) and Notebook 03 (merged onto RFM table for A/B test assignment).

In [ ]:
rfm.reset_index()[['CustomerID', 'Cluster']].to_csv('rfm_clusters.csv', index=False)
print("Exported rfm_clusters.csv")

---

## 3. Cohort Retention Analysis

**Question:** When do we lose customers — is it concentrated in the first few months, or spread evenly over time?

This matters because the answer determines the type of intervention. If most churn happens immediately after the first purchase, the problem is *activation* — converting a one-time buyer into a repeat customer. If churn is gradual, the problem is *long-term engagement*. The two require different strategies.

**Definition:** Retention rate for cohort X at month M = (distinct customers from X who purchased in month M) ÷ (total customers in cohort X).

In [ ]:
# ── Build cohort structure ─────────────────────────────────────
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')
df['CohortMonth'] = df.groupby('CustomerID')['InvoiceDate'].transform('min').dt.to_period('M')

def month_diff(d1, d2):
    return (d1.year - d2.year) * 12 + (d1.month - d2.month)

df['CohortIndex'] = df.apply(lambda x: month_diff(x['InvoiceMonth'], x['CohortMonth']), axis=1)

In [ ]:
# ── Retention matrix ───────────────────────────────────────────
cohort_data = df.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'].nunique().reset_index()
cohort_pivot = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')
cohort_size = cohort_pivot.iloc[:, 0]
retention = cohort_pivot.divide(cohort_size, axis=0)

In [ ]:
# ── Heatmap ────────────────────────────────────────────────────
plot_data = retention.iloc[:, 1:]  # exclude Month 0 (always 100%)

fig, ax = plt.subplots(figsize=(13, 7))
sns.heatmap(
    plot_data, ax=ax,
    cmap='Blues', annot=True, fmt='.0%',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Retention Rate'},
    mask=plot_data.isnull(),
    vmin=0, vmax=0.5
)
ax.set_title('Cohort Retention Analysis', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Months Since First Purchase', fontsize=11, labelpad=8)
ax.set_ylabel('Cohort (First Purchase Month)', fontsize=11, labelpad=8)
plt.tight_layout()
plt.savefig('cohort_retention.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

### Findings

**Early drop-off is the dominant pattern.** Across all cohorts, Month 1 retention is 15–30% — meaning 70–85% of customers do not return after their first purchase. This is not a seasonal anomaly; it is consistent across every cohort.

**Churn is front-loaded, not gradual.** The steepest decline happens between Month 0 and Month 1. From Month 2 onward, the rate of decline slows — customers who survive the first repeat purchase are meaningfully more likely to keep returning.

**Intermittent purchasing is common.** Several cohorts show Month 3 > Month 2, suggesting customers buy on irregular cycles rather than at fixed intervals. This is important context: a customer who skips a month is not necessarily churning.

**Retention patterns are stable across cohorts.** No single acquisition month is an outlier. The drop-off is a structural characteristic of this customer base.

> **What this means for the next section:** The data now tells us two things — *who* the valuable customers are (Section 2) and *when* we lose them (mostly after the first purchase). The remaining question is *how concentrated* the revenue impact is. If a small group drives most revenue, then even a modest improvement in early retention could have outsized financial impact → Section 4.

> **Downstream connection:** Notebook 02 quantifies the exact 1→2 purchase drop-off at 34.4% and measures the inter-purchase gap distribution to determine *when* a coupon intervention should arrive.

---

## 4. Revenue Concentration (Pareto Analysis)

**Question:** Is revenue evenly distributed across customers, or does a small group drive the majority of business value?

The RFM profiles above suggest concentration (C2 and C3 have much higher spend), but the Pareto analysis makes it precise — and determines whether the classic 80/20 rule applies here.

In [ ]:
# ── Cumulative revenue curve ───────────────────────────────────
rfm_sorted = rfm.sort_values('Monetary', ascending=False).copy()
rfm_sorted['cum_revenue_pct'] = rfm_sorted['Monetary'].cumsum() / rfm_sorted['Monetary'].sum()
rfm_sorted['cum_customer_pct'] = np.arange(1, len(rfm_sorted) + 1) / len(rfm_sorted)

# Find the 80% threshold
threshold_row = rfm_sorted[rfm_sorted['cum_revenue_pct'] <= 0.80]
pct_customers_80 = threshold_row['cum_customer_pct'].iloc[-1]
print(f"Top {pct_customers_80:.1%} of customers contribute 80% of total revenue")

In [ ]:
# ── Pareto curve with annotation ───────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5.5))

ax.plot(rfm_sorted['cum_customer_pct'], rfm_sorted['cum_revenue_pct'],
        color=PALETTE[0], linewidth=2.2, label='Actual')
ax.plot([0, 1], [0, 1], linestyle=':', color='#aaaaaa', linewidth=1, label='Perfect equality')

# 80% revenue threshold lines
ax.axhline(y=0.8, linestyle='--', color=PALETTE[3], alpha=0.5, linewidth=1)
ax.axvline(x=pct_customers_80, linestyle='--', color=PALETTE[3], alpha=0.5, linewidth=1)

# Annotation
ax.annotate(f'{pct_customers_80:.0%} of customers → 80% of revenue',
            xy=(pct_customers_80, 0.8), xytext=(0.45, 0.65),
            fontsize=10, color=PALETTE[3],
            arrowprops=dict(arrowstyle='->', color=PALETTE[3], lw=1.5),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=PALETTE[3], alpha=0.8))

# Classic 80/20 reference
ax.axvline(x=0.2, linestyle=':', color='#aaaaaa', alpha=0.4, linewidth=1)
ax.text(0.21, 0.05, '80/20 rule', fontsize=8, color='#aaaaaa', style='italic')

ax.set_xlabel("Cumulative % of Customers (ranked by spend)")
ax.set_ylabel("Cumulative % of Revenue")
ax.set_title("Revenue Concentration (Pareto Analysis)")
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
plt.tight_layout()
plt.show()

### Sensitivity Analysis — Excluding B2B accounts (C2)

C2 (13 customers) has avg 82 orders and avg £8.5k+ spend — far outside the retail norm. Including them may inflate the headline concentration figure. Let's re-run on retail-only customers (C0, C1, C3):

In [ ]:
# ── Sensitivity: retail-only ───────────────────────────────────
rfm_retail = rfm[rfm['Cluster'] != 2].copy()
rfm_retail_sorted = rfm_retail.sort_values('Monetary', ascending=False)
rfm_retail_sorted['cum_revenue_pct'] = rfm_retail_sorted['Monetary'].cumsum() / rfm_retail_sorted['Monetary'].sum()

top_80_retail = rfm_retail_sorted[rfm_retail_sorted['cum_revenue_pct'] <= 0.80]
pct_retail = top_80_retail.shape[0] / len(rfm_retail_sorted)

print(f"All customers:    top {pct_customers_80:.1%} → 80% of revenue")
print(f"Retail only:      top {pct_retail:.1%} → 80% of revenue")
print(f"Difference:       {(pct_retail - pct_customers_80)*100:+.1f} pp")

### Findings

**26% of customers contribute 80% of total revenue** — more dispersed than the classic 80/20 rule, but still clearly concentrated. Excluding B2B accounts shifts this to 31%, confirming the pattern is not driven by outliers.

| Scope | Top X% → 80% of revenue |
|-------|------------------------|
| All customers (incl. C2 Whales) | 26% |
| Retail only (excl. C2) | ~31% |

**Interpretation:**

The top tier (C2 + C3) is small in headcount but outsized in value — losing even one Whale is immediately visible in revenue. C3 loyals are the most scalable high-value segment; growing this group is the highest-ROI retention goal.

The 26% threshold (wider than a typical 10–20%) means there is a broad, upgradable mid-tier (C0). This is a growth opportunity: if even a fraction of C0 customers can be nudged toward C3 behaviour, the revenue impact is significant.

**The strategic conclusion is robust across both scenarios:** revenue is concentrated, the mid-tier is broad and upgradable, and the top tier warrants disproportionate retention investment.

> **Downstream connection:** This concentration finding, combined with the early churn pattern from Section 3, points to a clear intervention strategy: target the 1→2 purchase transition for C0 and C1 with a segment-specific coupon campaign. Notebook 02 calculates the precise timing windows; Notebook 03 simulates the A/B experiment.

---

## Summary

Four analyses, one consistent picture:

| Analysis | Key finding | Implication |
|----------|------------|-------------|
| **RFM Segmentation** | 4 distinct behavioural archetypes; C2+C3 are high-value and active, C0 is mid-cycle, C1 has structurally lapsed | A single tactic applied to all customers is inefficient — each segment needs a different approach |
| **Cohort Retention** | 70–85% of customers don't return after their first purchase; drop-off is front-loaded | The 1→2 purchase transition is the highest-leverage intervention point |
| **Revenue Concentration** | 26% of customers (31% retail-only) drive 80% of revenue | Retention has structurally higher ROI than acquisition; protecting C3 and upgrading C0 are the priority levers |

**The through-line:** Customer behaviour is not uniform, and the 1→2 purchase transition is the single highest-leverage point in the customer lifecycle. The remaining question is *how precisely* to intervene — which is the subject of Notebooks 02 and 03.

**What comes next:**

- **Notebook 02** uses DuckDB SQL to quantify the purchase funnel drop-off, measure inter-purchase gap distributions by segment, and produce the data-grounded rationale for segment-specific coupon windows (52 days for C0, 45 days for C1).
- **Notebook 03** simulates a controlled A/B experiment using those empirical baselines, validates statistical significance, and estimates combined revenue impact.

> The `rfm_clusters.csv` file exported above is the linking key that connects all three notebooks. Cluster labels C0–C3 are used consistently throughout.

---

## 5. Data Export for Tableau

Three files are exported in long format for dashboard visualisation in Tableau Public:

| File | Contents | Dashboard use |
|------|----------|---------------|
| `rfm_clusters.csv` | CustomerID → Cluster mapping | Segment filter |
| `cohort_retention.csv` | Retention rate by cohort × month | Cohort heatmap |
| `pareto_data.csv` | Cumulative revenue % (B2B incl. & excl.) | Pareto curve comparison |

In [ ]:
# ── Cohort retention → long format ─────────────────────────────
retention_long = retention.reset_index().melt(
    id_vars='CohortMonth', var_name='MonthIndex', value_name='RetentionRate'
)
retention_long['MonthIndex'] = retention_long['MonthIndex'].astype(int)
retention_long.to_csv('cohort_retention.csv', index=False)
print(f"cohort_retention.csv: {retention_long.shape[0]:,} rows")

In [ ]:
# ── Pareto data (incl. & excl. B2B) ───────────────────────────
rfm_sorted_full = rfm.sort_values('Monetary', ascending=False).reset_index()
rfm_sorted_full['cum_revenue_pct'] = rfm_sorted_full['Monetary'].cumsum() / rfm_sorted_full['Monetary'].sum()
rfm_sorted_full['cum_customer_pct'] = (rfm_sorted_full.index + 1) / len(rfm_sorted_full)
rfm_sorted_full['scope'] = 'Including B2B'

rfm_excl = rfm[rfm['Cluster'] != 2].sort_values('Monetary', ascending=False).reset_index()
rfm_excl['cum_revenue_pct'] = rfm_excl['Monetary'].cumsum() / rfm_excl['Monetary'].sum()
rfm_excl['cum_customer_pct'] = (rfm_excl.index + 1) / len(rfm_excl)
rfm_excl['scope'] = 'Excluding B2B'

pareto_data = pd.concat([rfm_sorted_full, rfm_excl], ignore_index=True)
pareto_data[['CustomerID', 'Monetary', 'Cluster', 'cum_revenue_pct', 'cum_customer_pct', 'scope']].to_csv('pareto_data.csv', index=False)
print(f"pareto_data.csv: {pareto_data.shape[0]:,} rows")